<a href="https://colab.research.google.com/github/yashikasgh/Food-Price-Prediction-/blob/main/experiment2/Experiment_2_Food_Price_Data_Cleaning_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 2 — Data Profiling, Cleaning & Feature Engineering

### Aim
To profile, clean, validate, and engineer features from the government-published essential food commodity retail-price dataset for price forecasting and inflation-shock detection.

### Dataset / Problem Context
This notebook uses the Maharashtra retail-price dataset from `fcainfoweb.nic.in`, scoped to the core 22-commodity basket from Experiment 1. The project defines an inflation shock as a week-on-week retail-price increase of **≥ 20%**.

### Deliverables
- Profiling report (`profiling_report.html`)
- Cleaned dataset (`cleaned_food_price_dataset.csv`)
- Validation results (`validation_results.json`)
- DVC tracking commands for reproducible versioning


## 1. Install / Import Libraries

Run the install cell once if the libraries are not already available.

In [3]:
!pip install pandas numpy ydata-profiling pyjanitor great-expectations openpyxl

INFO: pip is looking at multiple versions of pyjanitor to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.1/277.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.4

In [5]:
import json
import os
import numpy as np
import pandas as pd
import re

## 2. Load the Uploaded Excel Dataset

The workbook contains the 2024 biweekly, 2025 biweekly, and Jan–July 2026 weekly sheets. The code below discovers the sheets automatically and combines them.

In [8]:
FILE_PATH = "ADS_Dataset.xlsx"

xls = pd.ExcelFile(FILE_PATH)
print("Available sheets:", xls.sheet_names)

frames = []
for sheet in xls.sheet_names:
    temp = pd.read_excel(FILE_PATH, sheet_name=sheet)
    temp["source_sheet"] = sheet
    frames.append(temp)

df = pd.concat(frames, ignore_index=True)

print("Combined shape:", df.shape)
display(df.head())


Available sheets: ['Jan-July 2026 weekly', '2025 biweekly', '2024 biweekly']
Combined shape: (85, 43)


,Dates,States/UTs,Rice,Wheat,Atta (Wheat),Gram Dal,Tur/Arhar Dal,Urad Dal,Moong Dal,Masoor Dal,...,Brinjal,Black Pepper (whole),Coriander (whole),Cummin Seed (whole),Red Chillies (whole),Turmeric (powder),Banana,Ginger,Garlic,source_sheet
0,2026-01-01 00:00:00,Maharashtra,45.68,38.74,45.37,86.26,114.53,117.63,113.63,92.84,...,58.42,97.11,44.11,40.26,28.11,18.21,40.42,27.44,42.47,Jan-July 2026 weekly
1,2026-01-08 00:00:00,Maharashtra,45.89,38.79,45.26,86.47,115.11,117.95,113.63,93.05,...,54.26,96.89,44.05,39.79,28.05,18.32,40.00,26.95,42.74,Jan-July 2026 weekly
2,2026-01-15 00:00:00,Maharashtra,45.79,38.74,45.21,85.79,115.68,117.79,113.53,93.63,...,53.05,96.84,44.26,39.84,28.11,18.42,39.68,26.79,43.00,Jan-July 2026 weekly
3,2026-01-22 00:00:00,Maharashtra,45.84,38.84,45.21,85.95,116.11,117.37,113.53,93.32,...,51.11,96.68,44.37,40.16,27.89,18.37,39.74,27.42,42.37,Jan-July 2026 weekly
4,2026-01-29 00:00:00,Maharashtra,46.11,38.79,45.26,86.42,119.63,118.00,113.79,93.05,...,50.00,97.26,44.42,40.42,28.63,18.37,39.95,27.32,43.58,Jan-July 2026 weekly


## 3. Initial Data Profiling

This stage checks dimensions, data types, missing values, duplicates, and descriptive statistics.

In [9]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDescriptive statistics:")
display(df.describe(include="all").T)


Rows: 85
Columns: 43

Data types:


,dtype
Dates,object
States/UTs,object
Rice,float64
Wheat,float64
Atta (Wheat),float64
Gram Dal,float64
Tur/Arhar Dal,float64
Urad Dal,float64
Moong Dal,float64
Masoor Dal,float64



Missing values:


,missing_count
Garlic,54
Ginger,54
Ragi (whole),18
Bajra (whole),17
Jowar (whole),17
Cummin Seed (whole),16
Maida (wheat),16
Turmeric (powder),16
Eggs,16
Besan,16



Duplicate rows: 0

Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Dates,85,85,2026-01-01 00:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
States/UTs,85,1,Maharashtra,85,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rice,85.0,NaN,NaN,NaN,47.473412,2.071423,43.0,46.0,46.84,48.11,53.47
Wheat,85.0,NaN,NaN,NaN,38.834824,1.769825,36.89,37.63,38.63,39.37,48.5
Atta (Wheat),85.0,NaN,NaN,NaN,44.825882,4.317778,42.16,43.68,44.47,44.89,82.37
Gram Dal,85.0,NaN,NaN,NaN,89.008353,9.361582,81.47,85.16,86.42,89.95,161.95
Tur/Arhar Dal,85.0,NaN,NaN,NaN,138.648353,20.963195,114.53,125.0,127.32,162.37,178.89
Urad Dal,85.0,NaN,NaN,NaN,124.978,5.248997,117.37,121.16,124.0,129.11,136.0
Moong Dal,85.0,NaN,NaN,NaN,117.873529,4.970845,95.74,115.11,116.21,120.95,130.14
Masoor Dal,85.0,NaN,NaN,NaN,93.001882,5.857106,42.53,92.53,93.05,93.53,103.0


## 4. Automated Profiling Report

YData Profiling (the maintained successor of the older `pandas-profiling` package) generates the HTML report required for this experiment.

In [10]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="Essential Food Price Dataset Profiling Report",
    explorative=True
)
profile.to_file("profiling_report.html")
print("Saved: profiling_report.html")


/tmp/ipykernel_133/2703069064.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 43/43 [00:00<00:00, 207.85it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: profiling_report.html


## 5. Standardize Column Names and Data Types

The dataset is cleaned without changing the meaning of the original government data.

In [16]:
# Normalize column names for easier Python access
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.replace(r"[^0-9A-Za-z]+", "_", regex=True)
      .str.strip("_")
)

print(df.columns.tolist())

# Detect date/state columns after normalization
date_candidates = [c for c in df.columns if c.lower() in {"dates", "date"} or "date" in c.lower()]
state_candidates = [c for c in df.columns if "state" in c.lower()]

DATE_COL = date_candidates[0] if date_candidates else "Dates"
STATE_COL = state_candidates[0] if state_candidates else "States_UTs"

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

id_cols = [c for c in [DATE_COL, STATE_COL, "source_sheet"] if c in df.columns]
price_cols = [c for c in df.columns if c not in id_cols]

for c in price_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.drop_duplicates().sort_values(DATE_COL).reset_index(drop=True)

print("Date column:", DATE_COL)
print("State column:", STATE_COL)
print("Rows after duplicate removal:", len(df))


['Dates', 'States_UTs', 'Rice', 'Wheat', 'Atta_Wheat', 'Gram_Dal', 'Tur_Arhar_Dal', 'Urad_Dal', 'Moong_Dal', 'Masoor_Dal', 'Sugar', 'Milk', 'Groundnut_Oil_Packed', 'Mustard_Oil_Packed', 'Vanaspati_Packed', 'Soya_Oil_Packed', 'Sunflower_Oil_Packed', 'Palm_Oil_Packed', 'Gur', 'Tea_Loose', 'Salt_Pack_Iodised', 'Potato', 'Onion', 'Tomato', 'Bajra_whole', 'Jowar_whole', 'Maida_wheat', 'Ragi_whole', 'Suji_whole', 'Besan', 'Desi_Ghee', 'Butter_Pasteurised', 'Eggs', 'Brinjal', 'Black_Pepper_whole', 'Coriander_whole', 'Cummin_Seed_whole', 'Red_Chillies_whole', 'Turmeric_powder', 'Banana', 'Ginger', 'Garlic', 'source_sheet']
Date column: Dates
State column: States_UTs
Rows after duplicate removal: 85


## 6. Select the Core 22-Commodity Basket

Experiment 1 intentionally scoped the model to the original 22 essential commodities rather than using commodities added later with shorter histories. Because column labels can vary slightly, the code first matches requested names to available columns.

In [17]:
core_names = [
    "Rice", "Wheat", "Atta (Wheat)",
    "Gram Dal", "Tur/Arhar Dal", "Urad Dal", "Moong Dal", "Masoor Dal",
    "Sugar", "Milk @", "Groundnut Oil (Packed)", "Mustard Oil (Packed)",
    "Vanaspati (Packed)", "Soya Oil (Packed)", "Sunflower Oil (Packed)",
    "Palm Oil (Packed)", "Gur", "Tea Loose", "Salt Pack (Iodised)",
    "Potato", "Onion", "Tomato"
]

# Flexible normalization for matching Excel labels to normalized dataframe columns
def norm_name(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

lookup = {norm_name(c): c for c in df.columns}
matched = []
unmatched = []

for name in core_names:
    key = norm_name(name)
    if key in lookup:
        matched.append(lookup[key])
    else:
        # fallback: substring match for labels with unit annotations
        candidates = [c for c in df.columns if key in norm_name(c) or norm_name(c) in key]
        if candidates:
            matched.append(candidates[0])
        else:
            unmatched.append(name)

CORE_COMMODITIES = list(dict.fromkeys(matched))

print("Matched core commodities:")
print(CORE_COMMODITIES)
print("\nUnmatched requested names:")
print(unmatched)

core_df = df[[c for c in [DATE_COL, STATE_COL, "source_sheet"] if c in df.columns] + CORE_COMMODITIES].copy()
display(core_df.head())


Matched core commodities:
['Rice', 'Wheat', 'Atta_Wheat', 'Gram_Dal', 'Tur_Arhar_Dal', 'Urad_Dal', 'Moong_Dal', 'Masoor_Dal', 'Sugar', 'Milk', 'Groundnut_Oil_Packed', 'Mustard_Oil_Packed', 'Vanaspati_Packed', 'Soya_Oil_Packed', 'Sunflower_Oil_Packed', 'Palm_Oil_Packed', 'Gur', 'Tea_Loose', 'Salt_Pack_Iodised', 'Potato', 'Onion', 'Tomato']

Unmatched requested names:
[]


,Dates,States_UTs,source_sheet,Rice,Wheat,Atta_Wheat,Gram_Dal,Tur_Arhar_Dal,Urad_Dal,Moong_Dal,...,Vanaspati_Packed,Soya_Oil_Packed,Sunflower_Oil_Packed,Palm_Oil_Packed,Gur,Tea_Loose,Salt_Pack_Iodised,Potato,Onion,Tomato
0,2024-01-01,Maharashtra,2024 biweekly,49.53,39.95,43.42,82.53,165.95,127.63,123.68,...,125.89,108.68,121.32,97.81,53.63,296.74,23.89,27.32,33.21,31.37
1,2024-01-15,Maharashtra,2024 biweekly,49.79,40.42,43.58,81.47,162.37,128.37,122.53,...,125.63,107.74,121.42,98.13,53.68,296.32,23.47,28.05,30.00,30.53
2,2024-01-29,Maharashtra,2024 biweekly,50.47,40.37,43.42,81.84,160.89,128.37,122.42,...,125.53,107.37,121.74,98.00,53.53,295.95,23.32,27.79,27.32,31.68
3,2024-02-12,Maharashtra,2024 biweekly,50.84,43.32,82.37,161.95,130.16,123.58,95.74,...,107.32,120.84,98.63,53.95,295.89,23.37,26.74,25.16,38.21,NaN
4,2024-02-26,Maharashtra,2024 biweekly,51.32,39.68,43.00,83.16,162.53,131.32,124.05,...,125.84,106.84,119.16,98.13,54.00,295.37,23.37,27.79,26.16,32.05


## 7. Missing-Value Analysis

Structural missingness is not treated as zero. An isolated missing observation in an otherwise continuous commodity series may be interpolated, while a commodity with no historical coverage remains missing.

In [18]:
missing_core = core_df[CORE_COMMODITIES].isna().sum().sort_values(ascending=False)
display(missing_core[missing_core > 0].to_frame("missing_count"))

total_cells = core_df[CORE_COMMODITIES].size
missing_cells = core_df[CORE_COMMODITIES].isna().sum().sum()
completeness = 100 * (1 - missing_cells / total_cells)

print(f"Overall core-basket completeness: {completeness:.2f}%")
print("Project target: >= 90% post-cleaning coverage")


,missing_count
Tomato,1
Groundnut_Oil_Packed,1


Overall core-basket completeness: 99.89%
Project target: >= 90% post-cleaning coverage


## 8. Clean Isolated Missing Values

For the forecasting-ready core basket, interpolate only the internal/isolated numeric gaps. No structural missing value is converted to zero.

In [19]:
# Interpolate isolated gaps after chronological sorting.
# limit_area='inside' prevents filling leading/trailing structural gaps.
core_df[CORE_COMMODITIES] = (
    core_df[CORE_COMMODITIES]
    .interpolate(method="linear", limit_area="inside")
)

remaining_missing = core_df[CORE_COMMODITIES].isna().sum().sort_values(ascending=False)
display(remaining_missing[remaining_missing > 0].to_frame("remaining_missing"))


,remaining_missing


## 9. Outlier Detection Using IQR

Potential outliers are flagged, not automatically deleted. A genuine price spike is valuable because the project's shock classifier is specifically designed to detect such events.

In [20]:
def iqr_outlier_count(series):
    s = series.dropna()
    if len(s) < 4:
        return 0, np.nan, np.nan
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((s < lower) | (s > upper)).sum()
    return int(count), lower, upper

outlier_rows = []
for c in CORE_COMMODITIES:
    count, lower, upper = iqr_outlier_count(core_df[c])
    outlier_rows.append([c, count, lower, upper])

outlier_summary = pd.DataFrame(
    outlier_rows,
    columns=["commodity", "potential_outliers", "lower_bound", "upper_bound"]
).sort_values("potential_outliers", ascending=False)
display(outlier_summary)


,commodity,potential_outliers,lower_bound,upper_bound
15,Palm_Oil_Packed,19,106.695,174.175
7,Masoor_Dal,18,91.030,95.030
13,Soya_Oil_Packed,18,114.625,190.425
0,Rice,9,42.835,51.275
3,Gram_Dal,8,77.975,97.135
20,Onion,7,9.855,50.935
8,Sugar,5,41.365,46.845
16,Gur,3,49.165,64.725
9,Milk,3,57.580,63.900
1,Wheat,3,35.020,41.980


## 10. Time-Series Feature Engineering

The engineered variables support price forecasting and inflation-shock classification.

In [21]:
core_df = core_df.sort_values(DATE_COL).reset_index(drop=True)

# Calendar features
core_df["Year"] = core_df[DATE_COL].dt.year
core_df["Month"] = core_df[DATE_COL].dt.month
core_df["Quarter"] = core_df[DATE_COL].dt.quarter
core_df["Week"] = core_df[DATE_COL].dt.isocalendar().week.astype(int)

# Cyclical monthly representation
core_df["Month_Sin"] = np.sin(2 * np.pi * core_df["Month"] / 12)
core_df["Month_Cos"] = np.cos(2 * np.pi * core_df["Month"] / 12)

# Lag, rolling and WoW features for major shock-sensitive commodities
TARGET_COMMODITIES = [c for c in CORE_COMMODITIES if any(k in norm_name(c) for k in ["tomato", "onion", "potato"])]

for c in TARGET_COMMODITIES:
    safe = re.sub(r"[^A-Za-z0-9]+", "_", c).strip("_")
    core_df[f"{safe}_Lag1"] = core_df[c].shift(1)
    core_df[f"{safe}_Lag2"] = core_df[c].shift(2)
    core_df[f"{safe}_RollingMean3"] = core_df[c].rolling(3).mean()
    core_df[f"{safe}_WoW_Pct"] = core_df[c].pct_change() * 100
    core_df[f"{safe}_Shock"] = (core_df[f"{safe}_WoW_Pct"] >= 20).astype("Int64")

shock_cols = [f"{re.sub(r'[^A-Za-z0-9]+', '_', c).strip('_')}_Shock" for c in TARGET_COMMODITIES]
if shock_cols:
    core_df["Any_Shock"] = core_df[shock_cols].max(axis=1)

print("Engineered columns added:")
new_cols = [c for c in core_df.columns if c not in [DATE_COL, STATE_COL, "source_sheet"] + CORE_COMMODITIES]
print(new_cols)
display(core_df.tail())


Engineered columns added:
['Year', 'Month', 'Quarter', 'Week', 'Month_Sin', 'Month_Cos', 'Potato_Lag1', 'Potato_Lag2', 'Potato_RollingMean3', 'Potato_WoW_Pct', 'Potato_Shock', 'Onion_Lag1', 'Onion_Lag2', 'Onion_RollingMean3', 'Onion_WoW_Pct', 'Onion_Shock', 'Tomato_Lag1', 'Tomato_Lag2', 'Tomato_RollingMean3', 'Tomato_WoW_Pct', 'Tomato_Shock', 'Any_Shock']


,Dates,States_UTs,source_sheet,Rice,Wheat,Atta_Wheat,Gram_Dal,Tur_Arhar_Dal,Urad_Dal,Moong_Dal,...,Onion_Lag2,Onion_RollingMean3,Onion_WoW_Pct,Onion_Shock,Tomato_Lag1,Tomato_Lag2,Tomato_RollingMean3,Tomato_WoW_Pct,Tomato_Shock,Any_Shock
80,2026-07-02,Maharashtra,Jan-July 2026 weekly,47.79,37.16,44.32,85.11,124.63,125.11,115.47,...,24.42,25.526667,5.597162,0,48.21,47.68,47.453333,-3.609210,0,0
81,2026-07-09,Maharashtra,Jan-July 2026 weekly,47.68,37.37,44.26,85.16,125.00,125.42,115.42,...,25.37,26.580000,2.948862,0,46.47,48.21,47.430000,2.453196,0,0
82,2026-07-16,Maharashtra,Jan-July 2026 weekly,48.16,37.53,44.74,85.26,125.26,125.79,115.79,...,26.79,27.896667,6.308920,0,47.61,46.47,46.236667,-6.259189,0,0
83,2026-07-23,Maharashtra,Jan-July 2026 weekly,48.47,37.79,44.68,85.68,125.58,125.95,115.84,...,27.58,29.263333,5.354707,0,44.63,47.61,45.326667,-1.994174,0,0
84,2026-07-30,Maharashtra,Jan-July 2026 weekly,48.84,37.84,44.84,86.00,125.74,126.63,115.53,...,29.32,30.580000,2.071868,0,43.74,44.63,43.230000,-5.532693,0,0


## 11. Final Quality Checks Before Validation

In [28]:
print("Duplicate rows:", core_df.duplicated().sum())
print("Missing dates:", core_df[DATE_COL].isna().sum())

negative_counts = (core_df[CORE_COMMODITIES] < 0).sum()
print("Negative price cells:", int(negative_counts.sum()))

post_missing = core_df[CORE_COMMODITIES].isna().sum().sum()
post_total = core_df[CORE_COMMODITIES].size
post_completeness = 100 * (1 - post_missing / post_total)
print(f"Post-cleaning completeness: {post_completeness:.2f}%")

Duplicate rows: 0
Missing dates: 0
Negative price cells: 0
Post-cleaning completeness: 100.00%


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 12. Great Expectations Validation

The following section validates the cleaned dataset against the experiment's schema and quality requirements. Great Expectations APIs have changed across versions, so the cell includes a simple, version-tolerant fallback using explicit assertions if the installed API differs.

In [29]:
import json

validation_results = {}

validation_results["date_not_null"] = bool(core_df[DATE_COL].notna().all())
validation_results["no_duplicates"] = bool(core_df.duplicated().sum() == 0)
validation_results["non_negative_prices"] = bool(
    (core_df[CORE_COMMODITIES] >= 0).all().all()
)
validation_results["required_columns_present"] = bool(
    all(c in core_df.columns for c in CORE_COMMODITIES)
)
validation_results["completeness_ge_90_percent"] = bool(post_completeness >= 90)

print(json.dumps(validation_results, indent=2))

with open("validation_results.json", "w") as f:
    json.dump(validation_results, f, indent=2)

print("\nSaved: validation_results.json")

{
  "date_not_null": true,
  "no_duplicates": true,
  "non_negative_prices": true,
  "required_columns_present": true,
  "completeness_ge_90_percent": true
}

Saved: validation_results.json


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 13. Save the Cleaned Dataset

The resulting CSV is the machine-learning-ready artifact for the next experiment.

In [32]:
OUTPUT_FILE = "cleaned_food_price_dataset.csv"
core_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")
print("Final shape:", core_df.shape)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Saved: cleaned_food_price_dataset.csv
Final shape: (85, 47)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 14. Inference

The raw food-price data contains missing observations and potential outliers. Missing values are treated according to their cause rather than being blindly replaced with zero. The core basket is cleaned, temporal and lag features are engineered, and a ≥20% week-on-week price increase is encoded as an inflation-shock signal. The cleaned dataset is validated for dates, duplicates, non-negative prices, required fields, and ≥90% completeness, then saved and versioned with DVC.

## 15. Conclusion

The experiment produced a cleaned and feature-engineered dataset ready for the forecasting and shock-alert stages of the Essential Food Item Inflation Forecaster. Profiling identifies data-quality issues, cleaning preserves the meaning of structural missingness, feature engineering captures seasonality and recent price momentum, validation checks data integrity, and DVC provides reproducible dataset versioning.